In [1]:
import pandas as pd
import ast

df = pd.read_csv("phiusill_enriched_10000.csv")

df["ip_addresses"] = df["ip_addresses"].apply(ast.literal_eval)
df["name_servers"] = df["name_servers"].apply(ast.literal_eval)

print(df.shape)
df.head()

(10000, 61)


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label,is_phishing,domain,registered_domain,ip_addresses,name_servers
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,28,119,0,124,1,0,www.southbankmosaics.com,southbankmosaics.com,[142.93.145.212],"[ns2.namebrightdns.com, ns1.namebrightdns.com]"
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,8,39,0,217,1,0,www.uni-mainz.de,uni-mainz.de,[134.93.178.47],"[b.ns14.net, ns-extern.zdv.Uni-Mainz.DE, d.ns1..."
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,7,42,2,5,1,0,www.voicefmradio.co.uk,voicefmradio.co.uk,"[216.137.52.128, 216.137.52.12, 216.137.52.39,...","[ns2.livedns.co.uk, ns1.livedns.co.uk, ns3.liv..."
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,15,22,1,31,1,0,www.sfnmjournal.com,sfnmjournal.com,"[162.159.140.114, 172.66.0.112]","[ns1.reedelsevier.com, ns2.reedelsevier.com, n..."
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,34,72,1,85,1,0,www.rewildingargentina.org,rewildingargentina.org,"[172.67.178.236, 104.21.31.174]","[lovisa.ns.cloudflare.com, hans.ns.cloudflare...."


In [2]:
print("Shape:", df.shape)

print("\nColumns:")
for column in df.columns:
    print(column, end=", ")

Shape: (10000, 61)

Columns:
FILENAME, URL, URLLength, Domain, DomainLength, IsDomainIP, TLD, URLSimilarityIndex, CharContinuationRate, TLDLegitimateProb, URLCharProb, TLDLength, NoOfSubDomain, HasObfuscation, NoOfObfuscatedChar, ObfuscationRatio, NoOfLettersInURL, LetterRatioInURL, NoOfDegitsInURL, DegitRatioInURL, NoOfEqualsInURL, NoOfQMarkInURL, NoOfAmpersandInURL, NoOfOtherSpecialCharsInURL, SpacialCharRatioInURL, IsHTTPS, LineOfCode, LargestLineLength, HasTitle, Title, DomainTitleMatchScore, URLTitleMatchScore, HasFavicon, Robots, IsResponsive, NoOfURLRedirect, NoOfSelfRedirect, HasDescription, NoOfPopup, NoOfiFrame, HasExternalFormSubmit, HasSocialNet, HasSubmitButton, HasHiddenFields, HasPasswordField, Bank, Pay, Crypto, HasCopyrightInfo, NoOfImage, NoOfCSS, NoOfJS, NoOfSelfRef, NoOfEmptyRef, NoOfExternalRef, label, is_phishing, domain, registered_domain, ip_addresses, name_servers, 

In [3]:
ip_present = df["ip_addresses"].apply(len) > 0
ns_present = df["name_servers"].apply(len) > 0

df_graph = df[ip_present & ns_present].copy()

In [45]:
ml_features = [
    "URLLength",
    "DomainLength",
    "IsDomainIP",
    "URLSimilarityIndex",
    "CharContinuationRate",
    "URLCharProb",
    "TLDLength",
    "NoOfSubDomain",
    "HasObfuscation",
    "NoOfObfuscatedChar",
    "ObfuscationRatio",
    "NoOfLettersInURL",
    "LetterRatioInURL",
    "NoOfDegitsInURL",
    "DegitRatioInURL",
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL",
    "SpacialCharRatioInURL",
    "IsHTTPS",
    "NoOfURLRedirect",
    "NoOfSelfRedirect",
    "NoOfSelfRef",
    "NoOfEmptyRef",
    "NoOfExternalRef"
]
X = df_graph[ml_features]
Y = df_graph["is_phishing"]

print("Feature matrix:", X.shape)
print("\nMissing values:")
print(X.isnull().sum())

Feature matrix: (7937, 26)

Missing values:
URLLength                     0
DomainLength                  0
IsDomainIP                    0
URLSimilarityIndex            0
CharContinuationRate          0
URLCharProb                   0
TLDLength                     0
NoOfSubDomain                 0
HasObfuscation                0
NoOfObfuscatedChar            0
ObfuscationRatio              0
NoOfLettersInURL              0
LetterRatioInURL              0
NoOfDegitsInURL               0
DegitRatioInURL               0
NoOfEqualsInURL               0
NoOfQMarkInURL                0
NoOfAmpersandInURL            0
NoOfOtherSpecialCharsInURL    0
SpacialCharRatioInURL         0
IsHTTPS                       0
NoOfURLRedirect               0
NoOfSelfRedirect              0
NoOfSelfRef                   0
NoOfEmptyRef                  0
NoOfExternalRef               0
dtype: int64


#### check class balance

In [5]:
print(df_graph.shape)
print(df_graph["is_phishing"].value_counts())

(7937, 61)
is_phishing
0    5915
1    2022
Name: count, dtype: int64


#### Train/Test Split

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    stratify=Y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training samples: 6349
Testing samples: 1588

Training class distribution:
is_phishing
0    4732
1    1617
Name: count, dtype: int64

Testing class distribution:
is_phishing
0    1183
1     405
Name: count, dtype: int64


In [7]:
print("\nTraining percentages:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting percentages:")
print(y_test.value_counts(normalize=True) * 100)


Training percentages:
is_phishing
0    74.531422
1    25.468578
Name: proportion, dtype: float64

Testing percentages:
is_phishing
0    74.496222
1    25.503778
Name: proportion, dtype: float64


### Random Forest prediction model

#### 1. Create the model

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

#### 2. Train the model

In [9]:
rf_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


#### 3. Make predictions

In [10]:
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)

In [11]:
print(rf_model.classes_)

[0 1]


#### 4. Evaluate the model

In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1
)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

Accuracy:  0.9994
Precision: 1.0000
Recall:    0.9975
F1-score:  0.9988


#### Testing why we got 100% scores
##### Check 1 — Are the same URLs appearing in train and test?

In [13]:
train_urls = set(df_graph.loc[X_train.index, "URL"])
test_urls = set(df_graph.loc[X_test.index, "URL"])

overlap = train_urls.intersection(test_urls)

print("Training URLs:", len(train_urls))
print("Testing URLs:", len(test_urls))
print("Overlapping URLs:", len(overlap))

Training URLs: 6349
Testing URLs: 1588
Overlapping URLs: 0


##### Check 2 — Are the same domains appearing in both?

In [14]:
train_domains = set(df_graph.loc[X_train.index, "registered_domain"])
test_domains = set(df_graph.loc[X_test.index, "registered_domain"])

domain_overlap = train_domains.intersection(test_domains)

print("Overlapping registered domains:", len(domain_overlap))

Overlapping registered domains: 64


##### Check 3 — Show the confusion matrix

In [15]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1183    0]
 [   1  404]]


##### Check 4 — See which features are dominating

In [16]:
feature_importance = pd.DataFrame({
    "feature": ml_features,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(feature_importance)

                       feature    importance
3           URLSimilarityIndex  3.011890e-01
25             NoOfExternalRef  2.255446e-01
23                 NoOfSelfRef  1.871978e-01
20                     IsHTTPS  8.316852e-02
24                NoOfEmptyRef  4.266997e-02
13             NoOfDegitsInURL  4.266394e-02
18  NoOfOtherSpecialCharsInURL  2.609280e-02
14             DegitRatioInURL  2.427376e-02
19       SpacialCharRatioInURL  1.920969e-02
12            LetterRatioInURL  1.036339e-02
0                    URLLength  8.574793e-03
11            NoOfLettersInURL  8.292539e-03
4         CharContinuationRate  7.298942e-03
5                  URLCharProb  5.775852e-03
7                NoOfSubDomain  3.080169e-03
1                 DomainLength  2.813277e-03
6                    TLDLength  1.077605e-03
16              NoOfQMarkInURL  3.705847e-04
15             NoOfEqualsInURL  2.020414e-04
21             NoOfURLRedirect  1.165325e-04
22            NoOfSelfRedirect  2.425770e-05
8         

In [17]:
print("Duplicate feature rows:", X.duplicated().sum())

Duplicate feature rows: 1


In [18]:
feature_label_df = X.copy()
feature_label_df["is_phishing"] = Y.values

duplicates = feature_label_df[
    feature_label_df.duplicated(
        subset=ml_features,
        keep=False
    )
].sort_values(ml_features)

print(duplicates.shape)

(2, 27)


In [19]:
print(
    df_graph.groupby("is_phishing")[
        ["URLSimilarityIndex", "NoOfSelfRef"]
    ].describe()
)

            URLSimilarityIndex                                                 \
                         count        mean        std         min         25%   
is_phishing                                                                     
0                       5915.0  100.000000   0.000000  100.000000  100.000000   
1                       2022.0   56.051977  22.041094    0.155574   41.600291   

                                           NoOfSelfRef              \
                    50%         75%    max       count        mean   
is_phishing                                                          
0            100.000000  100.000000  100.0      5915.0  112.861031   
1             59.421702   71.993197  100.0      2022.0    0.438675   

                                                         
                    std  min   25%   50%    75%     max  
is_phishing                                              
0            136.986745  0.0  36.0  78.0  142.0  2651.0  
1          

### prepare the graph

In [20]:
import networkx as nx

G = nx.Graph()

#### 1. Add URL nodes

In [21]:
for idx, row in df_graph.iterrows():

    url_node = f"url_{idx}"

    G.add_node(
        url_node,
        node_type="url",
        url=row["URL"],
        is_phishing=row["is_phishing"]
    )

#### 2. Add domain nodes

In [22]:
for idx, row in df_graph.iterrows():

    url_node = f"url_{idx}"

    domain = row["registered_domain"]

    if pd.notna(domain) and domain != "":
        
        domain_node = f"domain_{domain}"

        G.add_node(
            domain_node,
            node_type="domain",
            value=domain
        )

        G.add_edge(
            url_node,
            domain_node,
            relationship="HAS_DOMAIN"
        )

#### 3. Add IP address nodes

In [23]:
for idx, row in df_graph.iterrows():

    url_node = f"url_{idx}"

    for ip in row["ip_addresses"]:

        if pd.notna(ip) and ip != "":

            ip_node = f"ip_{ip}"

            G.add_node(
                ip_node,
                node_type="ip",
                value=ip
            )

            G.add_edge(
                url_node,
                ip_node,
                relationship="RESOLVES_TO"
            )

#### 4. Add name-server nodes

In [24]:
for idx, row in df_graph.iterrows():

    url_node = f"url_{idx}"

    for ns in row["name_servers"]:

        if pd.notna(ns) and ns != "":

            ns_node = f"ns_{ns}"

            G.add_node(
                ns_node,
                node_type="name_server",
                value=ns
            )

            G.add_edge(
                url_node,
                ns_node,
                relationship="USES_NAME_SERVER"
            )

#### Check the graph

In [25]:
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

Number of nodes: 31111
Number of edges: 43017


In [26]:
from collections import Counter

node_types = Counter(
    data["node_type"]
    for _, data in G.nodes(data=True)
)

print("\nNode types:")
for node_type, count in node_types.items():
    print(f"{node_type}: {count}")


Node types:
url: 7937
domain: 6927
ip: 8374
name_server: 7873


In [27]:
edge_types = Counter(
    data["relationship"]
    for _, _, data in G.edges(data=True)
)

print("\nEdge types:")
for relationship, count in edge_types.items():
    print(f"{relationship}: {count}")


Edge types:
HAS_DOMAIN: 7937
RESOLVES_TO: 12612
USES_NAME_SERVER: 22468


#### Validate shared infrastructure relationships

In [28]:
# Check how many IP addresses are shared by multiple URLs

ip_usage = (
    df_graph["ip_addresses"]
    .explode()
    .dropna()
    .value_counts()
)

print("Unique IP addresses:", len(ip_usage))
print("IPs shared by 2+ URLs:", (ip_usage >= 2).sum())
print("IPs shared by 3+ URLs:", (ip_usage >= 3).sum())
print("Maximum URLs sharing one IP:", ip_usage.max())

Unique IP addresses: 8374
IPs shared by 2+ URLs: 708
IPs shared by 3+ URLs: 309
Maximum URLs sharing one IP: 336


In [29]:
# Check how many name servers are shared by multiple URLs

ns_usage = (
    df_graph["name_servers"]
    .explode()
    .dropna()
    .value_counts()
)

print("Unique name servers:", len(ns_usage))
print("Name servers shared by 2+ URLs:", (ns_usage >= 2).sum())
print("Name servers shared by 3+ URLs:", (ns_usage >= 3).sum())
print("Maximum URLs sharing one name server:", ns_usage.max())

Unique name servers: 7873
Name servers shared by 2+ URLs: 2821
Name servers shared by 3+ URLs: 1716
Maximum URLs sharing one name server: 188


#### Make URL nodes carry the Random Forest prediction

In [30]:
print(rf_model.classes_)

[0 1]


In [31]:
phishing_class_index = list(rf_model.classes_).index(1)

phishing_probability = y_proba[:, phishing_class_index]

In [32]:
# Store Random Forest predictions in the graph

for idx, prediction, probability in zip(
    X_test.index,
    y_pred,
    phishing_probability
):
    url_node = f"url_{idx}"

    if url_node in G.nodes:
        G.nodes[url_node]["prediction"] = int(prediction)
        G.nodes[url_node]["prediction_probability"] = float(probability)

In [33]:
# Check how many URL nodes have Random Forest predictions

predicted_urls = [
    node for node, data in G.nodes(data=True)
    if data.get("node_type") == "url"
    and "prediction" in data
]

print("URL nodes with RF predictions:", len(predicted_urls))

URL nodes with RF predictions: 1588


In [34]:
# Show a few predicted URL nodes

for node in predicted_urls[:5]:
    print(node, G.nodes[node])

url_3 {'node_type': 'url', 'url': 'https://www.sfnmjournal.com', 'is_phishing': 0, 'prediction': 0, 'prediction_probability': 0.0}
url_4 {'node_type': 'url', 'url': 'https://www.rewildingargentina.org', 'is_phishing': 0, 'prediction': 0, 'prediction_probability': 0.0}
url_5 {'node_type': 'url', 'url': 'https://www.globalreporting.org', 'is_phishing': 0, 'prediction': 0, 'prediction_probability': 0.0}
url_9 {'node_type': 'url', 'url': 'https://www.aap.org', 'is_phishing': 0, 'prediction': 0, 'prediction_probability': 0.0}
url_11 {'node_type': 'url', 'url': 'http://www.teramill.com', 'is_phishing': 1, 'prediction': 1, 'prediction_probability': 1.0}


#### Check the prediction distribution

In [35]:
prediction_counts = Counter(
    G.nodes[node]["prediction"]
    for node in predicted_urls
)

print("Random Forest predictions:")
print("Legitimate (0):", prediction_counts.get(0, 0))
print("Phishing (1):", prediction_counts.get(1, 0))

Random Forest predictions:
Legitimate (0): 1184
Phishing (1): 404


### BFS Traversal

#### Select a predicted phishing URL

In [36]:
# Get predicted phishing URLs

phishing_urls = [
    node for node, data in G.nodes(data=True)
    if data.get("node_type") == "url"
    and data.get("prediction") == 1
]

print("Predicted phishing URLs:", len(phishing_urls))

Predicted phishing URLs: 404


In [37]:
# Select the highest-confidence predicted phishing URL

target_url_node = max(
    phishing_urls,
    key=lambda node: G.nodes[node]["prediction_probability"]
)

target_data = G.nodes[target_url_node]

print("Target node:", target_url_node)
print("URL:", target_data["url"])
print("Prediction:", target_data["prediction"])
print(
    "Phishing probability:",
    round(target_data["prediction_probability"], 4)
)

Target node: url_11
URL: http://www.teramill.com
Prediction: 1
Phishing probability: 1.0


#### Run BFS

In [38]:
# BFS traversal from the target URL

max_depth = 2

bfs_distances = nx.single_source_shortest_path_length(
    G,
    target_url_node,
    cutoff=max_depth
)

print("Target URL:", target_data["url"])
print("Maximum BFS depth:", max_depth)
print("Nodes discovered:", len(bfs_distances))

Target URL: http://www.teramill.com
Maximum BFS depth: 2
Nodes discovered: 8


#### Inspect what BFS discovered

In [39]:
# Display nodes discovered by BFS

for node, distance in sorted(
    bfs_distances.items(),
    key=lambda x: (x[1], x[0])
):

    data = G.nodes[node]

    print(
        f"Hop {distance}: "
        f"{data.get('node_type')} → "
        f"{data.get('value', data.get('url', node))}"
    )

Hop 0: url → http://www.teramill.com
Hop 1: domain → teramill.com
Hop 1: ip → 91.239.200.44
Hop 1: name_server → ns1.thinline.cz
Hop 1: name_server → ns2.thinline.cz
Hop 1: name_server → ns3.cesky-hosting.eu
Hop 2: url → https://www.pamatkyaprirodakarlovarska.cz
Hop 2: url → https://www.ikaros.cz


### Evidence Extraction

#### Inspect the related URLs

In [40]:
# Inspect URL nodes discovered by BFS

print("Related URLs found by BFS:\n")

for node, distance in bfs_distances.items():

    data = G.nodes[node]

    if data.get("node_type") == "url" and distance > 0:

        print(f"Hop: {distance}")
        print(f"URL: {data.get('url')}")
        print(f"Actual label: {data.get('is_phishing')}")
        print(f"RF prediction: {data.get('prediction')}")
        print(
            f"RF phishing probability: "
            f"{data.get('prediction_probability')}"
        )
        print("-" * 60)

Related URLs found by BFS:

Hop: 2
URL: https://www.pamatkyaprirodakarlovarska.cz
Actual label: 0
RF prediction: 0
RF phishing probability: 0.0
------------------------------------------------------------
Hop: 2
URL: https://www.ikaros.cz
Actual label: 0
RF prediction: None
RF phishing probability: None
------------------------------------------------------------


#### Inspect the relationships themselves

In [41]:
# Show the graph paths from the target URL

for node, distance in bfs_distances.items():

    if node == target_url_node:
        continue

    path = nx.shortest_path(
        G,
        target_url_node,
        node
    )

    print(f"\nTarget → {node}")
    print(f"Distance: {distance}")
    print("Path:")

    for path_node in path:
        data = G.nodes[path_node]

        value = data.get(
            "value",
            data.get("url", path_node)
        )

        print(
            f"  {data.get('node_type')} → {value}"
        )


Target → domain_teramill.com
Distance: 1
Path:
  url → http://www.teramill.com
  domain → teramill.com

Target → ip_91.239.200.44
Distance: 1
Path:
  url → http://www.teramill.com
  ip → 91.239.200.44

Target → ns_ns1.thinline.cz
Distance: 1
Path:
  url → http://www.teramill.com
  name_server → ns1.thinline.cz

Target → ns_ns2.thinline.cz
Distance: 1
Path:
  url → http://www.teramill.com
  name_server → ns2.thinline.cz

Target → ns_ns3.cesky-hosting.eu
Distance: 1
Path:
  url → http://www.teramill.com
  name_server → ns3.cesky-hosting.eu

Target → url_1170
Distance: 2
Path:
  url → http://www.teramill.com
  name_server → ns1.thinline.cz
  url → https://www.pamatkyaprirodakarlovarska.cz

Target → url_9923
Distance: 2
Path:
  url → http://www.teramill.com
  name_server → ns2.thinline.cz
  url → https://www.ikaros.cz


#### create structured evidence

In [42]:
# URLs whose labels are available to the evidence extraction process

known_training_indices = set(X_train.index)

print("Known training URLs:", len(known_training_indices))

Known training URLs: 6349


In [43]:
# Classify evidence using only known training-set labels

for item in evidence:

    related_url = item["related_url"]

    # Direct evidence does not have a related URL
    if related_url is None:
        item["evidence_role"] = "context"
        continue

    # Find the graph node corresponding to the related URL
    related_node = None

    for node, data in G.nodes(data=True):

        if (
            data.get("node_type") == "url"
            and data.get("url") == related_url
        ):
            related_node = node
            break

    if related_node is None:
        item["evidence_role"] = "context"
        continue

    # Recover the original dataframe index
    related_index = int(
        related_node.replace("url_", "")
    )

    # Only training URLs are considered known evidence sources
    if related_index not in known_training_indices:
        item["evidence_role"] = "unavailable"

    elif G.nodes[related_node].get("is_phishing") == 1:
        item["evidence_role"] = "supporting"

    else:
        item["evidence_role"] = "neutral"

NameError: name 'evidence' is not defined

In [ ]:
for item in evidence:
    print(item)

#### Find a URL to support our evidance

In [ ]:
# Find predicted-phishing URLs with known phishing neighbours

candidate_targets = []

for target_node in phishing_urls:

    distances = nx.single_source_shortest_path_length(
        G,
        target_node,
        cutoff=2
    )

    phishing_neighbours = []

    for node, distance in distances.items():

        if distance != 2:
            continue

        data = G.nodes[node]

        if data.get("node_type") != "url":
            continue

        related_index = int(
            node.replace("url_", "")
        )

        # Only use training-set URLs as known evidence
        if (
            related_index in known_training_indices
            and data.get("is_phishing") == 1
        ):
            phishing_neighbours.append(node)

    if phishing_neighbours:
        candidate_targets.append(
            (target_node, phishing_neighbours)
        )

print(
    "Predicted-phishing targets with "
    "known phishing neighbours:",
    len(candidate_targets)
)

In [ ]:
for target_node, neighbours in candidate_targets[:10]:

    data = G.nodes[target_node]

    print("\nTarget:", data["url"])
    print(
        "RF phishing probability:",
        round(data["prediction_probability"], 4)
    )
    print(
        "Known phishing neighbours:",
        len(neighbours)
    )

In [ ]:
# Select a predicted-phishing URL for the evidence demonstration

target_url_node = next(
    node
    for node, neighbours in candidate_targets
    if G.nodes[node].get("url") == "http://nmve.xyz/"
)

target_data = G.nodes[target_url_node]

print("Target URL:", target_data["url"])
print("Actual label:", target_data["is_phishing"])
print("RF prediction:", target_data["prediction"])
print(
    "RF phishing probability:",
    target_data["prediction_probability"]
)

### Run BFS for that target

In [ ]:
# BFS traversal for the selected target

max_depth = 2

bfs_distances = nx.single_source_shortest_path_length(
    G,
    target_url_node,
    cutoff=max_depth
)

print("Target URL:", target_data["url"])
print("Maximum BFS depth:", max_depth)
print("Nodes discovered:", len(bfs_distances))

In [ ]:
from collections import Counter

bfs_node_types = Counter(
    G.nodes[node].get("node_type")
    for node in bfs_distances
)

print("\nBFS node types:")

for node_type, count in bfs_node_types.items():
    print(f"{node_type}: {count}")

In [ ]:
# Extract evidence candidates from BFS results

evidence = []

for node, distance in bfs_distances.items():

    if node == target_url_node:
        continue

    data = G.nodes[node]
    node_type = data.get("node_type")

    # Direct infrastructure connected to target
    if distance == 1 and node_type in {
        "ip",
        "name_server",
        "domain"
    }:

        evidence.append({
            "evidence_type": f"direct_{node_type}",
            "entity": data.get("value", node),
            "distance": distance,
            "related_url": None,
            "related_url_label": None
        })

    # Related URLs discovered through shared infrastructure
    elif distance == 2 and node_type == "url":

        path = nx.shortest_path(
            G,
            target_url_node,
            node
        )

        if len(path) == 3:

            relationship_node = path[1]
            relationship_data = G.nodes[relationship_node]

            evidence.append({
                "evidence_type": (
                    f"shared_{relationship_data.get('node_type')}"
                ),
                "entity": relationship_data.get(
                    "value",
                    relationship_node
                ),
                "distance": distance,
                "related_url": data.get("url"),
                "related_url_label": data.get(
                    "is_phishing"
                )
            })

In [ ]:
# Classify evidence using only known training-set labels

for item in evidence:

    if item["related_url"] is None:
        item["evidence_role"] = "context"
        continue

    related_node = None

    for node, data in G.nodes(data=True):

        if (
            data.get("node_type") == "url"
            and data.get("url") == item["related_url"]
        ):
            related_node = node
            break

    if related_node is None:
        item["evidence_role"] = "context"
        continue

    related_index = int(
        related_node.replace("url_", "")
    )

    if related_index not in known_training_indices:
        item["evidence_role"] = "unavailable"

    elif G.nodes[related_node].get("is_phishing") == 1:
        item["evidence_role"] = "supporting"

    else:
        item["evidence_role"] = "neutral"

In [ ]:
print("Evidence candidates:", len(evidence))

for item in evidence:
    print(item)

#### Aggregate the evidence

In [ ]:
from collections import defaultdict

aggregated_evidence = defaultdict(
    lambda: {
        "evidence_type": None,
        "entity": None,
        "distance": None,
        "known_phishing_urls": [],
        "known_legitimate_urls": []
    }
)

for item in evidence:

    # We only aggregate shared infrastructure evidence
    if not item["evidence_type"].startswith("shared_"):
        continue

    key = (
        item["evidence_type"],
        item["entity"]
    )

    record = aggregated_evidence[key]

    record["evidence_type"] = item["evidence_type"]
    record["entity"] = item["entity"]
    record["distance"] = item["distance"]

    if item["evidence_role"] == "supporting":
        record["known_phishing_urls"].append(
            item["related_url"]
        )

    elif item["evidence_role"] == "neutral":
        record["known_legitimate_urls"].append(
            item["related_url"]
        )

In [ ]:
aggregated_evidence = list(
    aggregated_evidence.values()
)

print(
    "Aggregated evidence:",
    len(aggregated_evidence)
)

In [ ]:
for item in aggregated_evidence:

    print("\nEvidence type:", item["evidence_type"])
    print("Entity:", item["entity"])
    print("Distance:", item["distance"])
    print(
        "Known phishing URLs:",
        len(item["known_phishing_urls"])
    )
    print(
        "Known legitimate URLs:",
        len(item["known_legitimate_urls"])
    )

### Evidence Ranking

#### define the relationship scores

In [ ]:
# Evidence scores from the proposed scoring schema

evidence_type_scores = {
    "shared_ip": 5,
    "shared_name_server": 4,
    "suspicious_url_keyword": 3,
    "shared_graph_entity": 2
}

hop_scores = {
    1: 2,
    2: 1
}

#### Calculate evidence scores

In [ ]:
# Combine direct evidence with aggregated shared evidence

direct_evidence = [
    item
    for item in evidence
    if item["evidence_type"].startswith("direct_")
]

rankable_evidence = (
    direct_evidence +
    aggregated_evidence
)

print("Direct evidence:", len(direct_evidence))
print("Aggregated shared evidence:", len(aggregated_evidence))
print("Total rankable evidence:", len(rankable_evidence))

In [ ]:
# Calculate scores for aggregated shared evidence

for item in rankable_evidence:

    evidence_type = item["evidence_type"]
    distance = item["distance"]

    phishing_count = len(
        item.get("known_phishing_urls", [])
    )

    score = 0

    # -----------------------------------------
    # Evidence type score
    # -----------------------------------------

    if evidence_type == "shared_ip":

        # +5 only when the IP is shared with
        # at least one known phishing URL
        if phishing_count > 0:
            score += 5

    elif evidence_type == "shared_name_server":

        score += 4

    # -----------------------------------------
    # Graph distance score
    # -----------------------------------------

    if distance == 1:
        score += 2

    elif distance == 2:
        score += 1

    item["score"] = score

#### rank the evidence

In [ ]:
# Rank evidence from highest to lowest score

ranked_evidence = sorted(
    rankable_evidence,
    key=lambda item: item["score"],
    reverse=True
)

In [ ]:
print("Ranked evidence:\n")

for rank, item in enumerate(
    ranked_evidence,
    start=1
):

    phishing_count = len(
        item.get("known_phishing_urls", [])
    )

    legitimate_count = len(
        item.get("known_legitimate_urls", [])
    )

    # Determine role for aggregated evidence
    if "evidence_role" in item:
        role = item["evidence_role"]

    elif phishing_count > 0:
        role = "supporting"

    elif legitimate_count > 0:
        role = "neutral"

    else:
        role = "context"

    print(
        f"{rank}. "
        f"{item['evidence_type']} | "
        f"{item['entity']} | "
        f"score={item['score']} | "
        f"distance={item['distance']} | "
        f"role={role} | "
        f"phishing_urls={phishing_count}"
    )

### Generate human-readable explanation

In [ ]:
def generate_explanation(
    target_data,
    ranked_evidence,
    max_supporting_evidence=3
):
    """
    Convert ranked graph evidence into a human-readable explanation.
    """

    prediction = target_data.get("prediction")
    probability = target_data.get(
        "prediction_probability"
    )
    url = target_data.get("url")

    # Convert prediction to readable label
    if prediction == 1:
        prediction_label = "Phishing"
    else:
        prediction_label = "Legitimate"

    # ---------------------------------------------------------
    # Start explanation
    # ---------------------------------------------------------

    explanation = []

    explanation.append(
        f"Prediction: {prediction_label}"
    )

    if probability is not None:
        explanation.append(
            f"Random Forest confidence: "
            f"{probability * 100:.1f}%"
        )

    # ---------------------------------------------------------
    # Select highest-ranked supporting evidence
    # ---------------------------------------------------------

    supporting_evidence = [
        item
        for item in ranked_evidence
        if (
            item.get("evidence_role") == "supporting"
            or (
                item.get("evidence_type") == "shared_ip"
                and len(
                    item.get(
                        "known_phishing_urls",
                        []
                    )
                ) > 0
            )
        )
    ]

    supporting_evidence = supporting_evidence[
        :max_supporting_evidence
    ]

    # ---------------------------------------------------------
    # Generate supporting evidence statements
    # ---------------------------------------------------------

    if supporting_evidence:

        explanation.append(
            "Supporting Evidence:"
        )

        for item in supporting_evidence:

            evidence_type = item["evidence_type"]
            entity = item["entity"]
            score = item["score"]

            phishing_count = len(
                item.get(
                    "known_phishing_urls",
                    []
                )
            )

            if evidence_type == "shared_ip":

                statement = (
                    f"- Shares IP address {entity} "
                    f"with {phishing_count} known "
                    f"phishing URL"
                )

                if phishing_count != 1:
                    statement += "s"

                statement += f" (evidence score: +{score})."

                explanation.append(statement)

            elif evidence_type == "shared_name_server":

                statement = (
                    f"- Shares authoritative name server "
                    f"{entity} "
                    f"(evidence score: +{score})."
                )

                explanation.append(statement)

    else:

        explanation.append(
            "Supporting Evidence: "
            "No strong supporting graph evidence "
            "was identified."
        )

    # ---------------------------------------------------------
    # Add a concise overall explanation
    # ---------------------------------------------------------

    if supporting_evidence:

        if prediction == 1:

            explanation.append(
                "Overall, the shared infrastructure "
                "relationships provide strong graph-based "
                "evidence supporting the phishing "
                "classification."
            )

        else:

            explanation.append(
                "Overall, the available graph relationships "
                "do not provide strong evidence of phishing."
            )

    return "\n".join(explanation)

In [ ]:
explanation = generate_explanation(
    target_data,
    ranked_evidence
)

print(explanation)

## Graph Visualisation for Report

A small subgraph is extracted from the complete heterogeneous graph to
illustrate how a predicted phishing URL is connected to its domain,
IP address, name servers, and other URLs through shared infrastructure.

In [46]:
import matplotlib.pyplot as plt
import networkx as nx


def create_report_graph(target_url):
    """
    Create a small graph suitable for inclusion in the report.

    The graph contains:
    - Target URL
    - Target domain
    - Target IP address
    - Target name servers
    - Two related URLs sharing the target IP
    """

    graph = graph_manager.get_graph()

    # --------------------------------------------------
    # Find target URL node
    # --------------------------------------------------

    target_node = None

    for node, data in graph.nodes(data=True):

        if (
            data.get("node_type") == "url"
            and data.get("url") == target_url
        ):
            target_node = node
            break

    if target_node is None:
        raise ValueError(
            f"URL not found in graph: {target_url}"
        )

    # --------------------------------------------------
    # Find target IP nodes
    # --------------------------------------------------

    target_ips = []

    for neighbour in graph.neighbors(target_node):

        data = graph.nodes[neighbour]

        if data.get("node_type") == "ip":
            target_ips.append(neighbour)

    if not target_ips:
        raise ValueError(
            "Target URL has no IP address in the graph."
        )

    # --------------------------------------------------
    # Select the first target IP
    # --------------------------------------------------

    shared_ip_node = target_ips[0]

    shared_ip = graph.nodes[
        shared_ip_node
    ]["value"]

    # --------------------------------------------------
    # Find other URLs sharing the same IP
    # --------------------------------------------------

    related_urls = []

    for neighbour in graph.neighbors(
        shared_ip_node
    ):

        data = graph.nodes[neighbour]

        if (
            data.get("node_type") == "url"
            and neighbour != target_node
        ):
            related_urls.append(neighbour)

    # Use only two related URLs
    related_urls = related_urls[:2]

    # --------------------------------------------------
    # Build small report graph
    # --------------------------------------------------

    report_graph = nx.Graph()

    selected_nodes = [
        target_node,
        shared_ip_node
    ]

    # Add target domain and name servers
    for neighbour in graph.neighbors(target_node):

        data = graph.nodes[neighbour]

        if data.get("node_type") in [
            "domain",
            "name_server"
        ]:

            selected_nodes.append(neighbour)

    # Add related URLs
    selected_nodes.extend(related_urls)

    # Remove duplicates
    selected_nodes = list(
        dict.fromkeys(selected_nodes)
    )

    # Add selected nodes
    for node in selected_nodes:

        report_graph.add_node(
            node,
            **graph.nodes[node]
        )

    # Add edges between selected nodes
    for source, target, data in graph.edges(
        selected_nodes,
        data=True
    ):

        if (
            source in report_graph
            and target in report_graph
        ):

            report_graph.add_edge(
                source,
                target,
                **data
            )

    # --------------------------------------------------
    # Draw graph
    # --------------------------------------------------

    plt.figure(
        figsize=(12, 7)
    )

    pos = nx.spring_layout(
        report_graph,
        seed=42,
        k=1.5
    )

    # Node groups
    node_groups = {
        "url": [],
        "domain": [],
        "ip": [],
        "name_server": []
    }

    for node, data in report_graph.nodes(
        data=True
    ):

        node_type = data.get(
            "node_type"
        )

        if node_type in node_groups:
            node_groups[node_type].append(
                node
            )

    # Draw nodes
    nx.draw_networkx_nodes(
        report_graph,
        pos,
        nodelist=node_groups["url"],
        node_color="lightblue",
        node_size=2200,
        node_shape="o"
    )

    nx.draw_networkx_nodes(
        report_graph,
        pos,
        nodelist=node_groups["domain"],
        node_color="lightgreen",
        node_size=1800,
        node_shape="s"
    )

    nx.draw_networkx_nodes(
        report_graph,
        pos,
        nodelist=node_groups["ip"],
        node_color="orange",
        node_size=2000,
        node_shape="D"
    )

    nx.draw_networkx_nodes(
        report_graph,
        pos,
        nodelist=node_groups["name_server"],
        node_color="plum",
        node_size=1800,
        node_shape="^"
    )

    # Draw edges
    nx.draw_networkx_edges(
        report_graph,
        pos,
        width=1.8
    )

    # --------------------------------------------------
    # Create readable labels
    # --------------------------------------------------

    labels = {}

    for node, data in report_graph.nodes(
        data=True
    ):

        node_type = data.get(
            "node_type"
        )

        if node_type == "url":

            url = data.get(
                "url",
                node
            )

            # Shorten long URLs for figure
            if len(url) > 28:
                url = url[:25] + "..."

            labels[node] = url

        else:

            labels[node] = data.get(
                "value",
                node
            )

    nx.draw_networkx_labels(
        report_graph,
        pos,
        labels=labels,
        font_size=8
    )

    plt.title(
        "Example Heterogeneous Graph for Phishing Evidence Extraction",
        fontsize=14,
        fontweight="bold"
    )

    plt.axis("off")

    plt.tight_layout()

    plt.show()

In [47]:
create_report_graph(
    "https://fb-restriction-case-97be5.web.app/"
)

NameError: name 'graph_manager' is not defined